# Wave-action and quadratic-energy spectra
Plot instantaneous frames or a pointwise average against wavenumber or angular frequency.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'gp2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from gp2d_plotting import (available_frames, curves_for_frames, density_in_frequency,
    parameter_float, positive_xy, read_csv, read_parameters, repository_root,
    save_figure, select_frames, spectral_abscissa, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
SPECTRA_FILE = ROOT / 'output/spectra.csv'  # CSV produced by the solver.
PARAMETER_FILE = ROOT / 'output/resolved_parameters.txt'  # Supplies c and mu for omega(k).
WAVE_ACTION_FIGURE = ROOT / 'figures/wave_action_spectrum.pdf'  # First PDF output.
ENERGY_FIGURE = ROOT / 'figures/energy_spectrum.pdf'  # Second PDF output.

# 'single': one curve and exactly one selected frame.
# 'multiple': one curve for every selected frame.
# 'average': one pointwise average over all selected frames.
MODE = 'single'

# Explicit frame numbers to plot. Negative indices count from the end, so [-1]
# means the most recent frame. Set this to None to use START/STOP/STRIDE below.
FRAMES = [-1]
FRAME_START = None  # First frame when FRAMES=None; None means the first available.
FRAME_STOP = None   # Last frame, inclusive; None means the last available.
FRAME_STRIDE = 1    # Keep every nth available frame in the selected range.

# 'wavenumber' plots against k. 'frequency' uses omega=|-c*k^2+mu|.
X_AXIS = 'wavenumber'
# For a frequency plot, True converts densities using n_omega=n_k/|d omega/d k|.
# This setting has no effect when X_AXIS='wavenumber'.
DENSITY_IN_FREQUENCY = True
# True selects the solver's running segment-mean columns instead of instantaneous data.
USE_SEGMENT_MEAN = False
LOG_SCALE = True    # True gives logarithmic x and y axes and omits non-positive points.
USE_TEX = True      # True uses an external LaTeX installation for all figure text.
FONT_SIZE = 16      # Base font size in points.

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = read_csv(SPECTRA_FILE)
frames = select_frames(available_frames(table), FRAMES, start=FRAME_START,
                       stop=FRAME_STOP, stride=FRAME_STRIDE)
wave_column = 'segment_mean_wave_action_spectrum' if USE_SEGMENT_MEAN else 'wave_action_spectrum'
energy_column = 'segment_mean_quadratic_energy_spectrum' if USE_SEGMENT_MEAN else 'quadratic_energy_spectrum'
k, curves = curves_for_frames(table, frames, [wave_column, energy_column], MODE)
parameters = read_parameters(PARAMETER_FILE) if PARAMETER_FILE.exists() else {}
coefficient = parameter_float(parameters, 'dispersionCoefficient', -1.0)
chemical_potential = parameter_float(parameters, 'chemicalPotential', 0.0)
x, x_label = spectral_abscissa(k, X_AXIS, coefficient, chemical_potential)
x_order = np.argsort(x)
print(f'Selected frames: {frames}')

In [ ]:
plot_curves = []
for curve in curves:
    wave = np.asarray(curve[wave_column])
    energy = np.asarray(curve[energy_column])
    if X_AXIS == 'frequency' and DENSITY_IN_FREQUENCY:
        wave = density_in_frequency(k, wave, coefficient)
        energy = density_in_frequency(k, energy, coefficient)
    plot_curves.append((curve['label'], wave, energy))

density_suffix = X_AXIS == 'frequency' and DENSITY_IN_FREQUENCY
wave_ylabel = r'$n_\omega$' if density_suffix else r'$n_k$'
energy_ylabel = r'$E_\omega^{(2)}$' if density_suffix else r'$E_k^{(2)}$'

## Wave-action spectrum

In [ ]:
fig, axis = plt.subplots(figsize=(7.0, 5.2))
for label, wave, _ in plot_curves:
    plot_x, plot_y = positive_xy(x[x_order], wave[x_order]) if LOG_SCALE else (x[x_order], wave[x_order])
    axis.plot(plot_x, plot_y, label=label)
axis.set_xlabel(x_label)
axis.set_ylabel(wave_ylabel)
if LOG_SCALE:
    axis.set_xscale('log')
    axis.set_yscale('log')
axis.grid(True, which='both', alpha=0.2)
axis.legend()
saved = save_figure(fig, WAVE_ACTION_FIGURE)
print(f'Wrote {saved}')
plt.show()

## Quadratic-energy spectrum

In [ ]:
fig, axis = plt.subplots(figsize=(7.0, 5.2))
for label, _, energy in plot_curves:
    plot_x, plot_y = positive_xy(x[x_order], energy[x_order]) if LOG_SCALE else (x[x_order], energy[x_order])
    axis.plot(plot_x, plot_y, label=label)
axis.set_xlabel(x_label)
axis.set_ylabel(energy_ylabel)
if LOG_SCALE:
    axis.set_xscale('log')
    axis.set_yscale('log')
axis.grid(True, which='both', alpha=0.2)
axis.legend()
saved = save_figure(fig, ENERGY_FIGURE)
print(f'Wrote {saved}')
plt.show()